# Module 6 · Project — Knowledge Assistant v3

**From 0 to Agentic AI — DataHack Summit 2026**

🧩 **v3 — internal knowledge.** v2 could search the web and draft actions, but it still
couldn't answer about *our* docs. Now we give it a **retrieval tool** over a company knowledge
base — the agent decides when to consult internal docs vs. the web. This breaks the
"no knowledge" wall from Module 2.

> 📝 **Your turn.** Cells marked **TODO** need implementing. Stuck? Peek at the `_SOLUTION`.

### What you'll build
1. A **vector store** over company docs (from the RAG demo)
2. A **`search_company_docs` tool** that retrieves + returns cited context
3. Add it alongside web search → the agent **routes** between internal and web knowledge

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml.
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" "langgraph>=1.0,<2" \
               "langchain-text-splitters>=0.3" "langchain-chroma>=0.2" "chromadb>=0.5" "seltz>=1.5.0"

In [ ]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY", "SELTZ_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · Build the knowledge base

Chunk + embed a few company docs into Chroma (same as the RAG demo).

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

docs = [
    Document(page_content="The billing service is owned by the Payments team. On-call lead: Sam. Escalate outages in #billing-oncall.", metadata={"source": "runbook", "team": "payments"}),
    Document(page_content="Employees get 28 days of paid time off (PTO) per year, plus public holidays. Requests go through the HR portal.", metadata={"source": "hr-policy", "team": "people"}),
    Document(page_content="Production deploys run weekdays at 9pm IST via the release bot. Rollbacks are one command: `deploy rollback <service>`.", metadata={"source": "runbook", "team": "platform"}),
    Document(page_content="Expense reports over $500 need manager approval before submission. Reimbursement takes 5-7 business days.", metadata={"source": "finance-policy", "team": "finance"}),
]
chunks = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30).split_documents(docs)
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("indexed", vectorstore._collection.count(), "chunks")

---
## Step 2 · The retrieval tool

Wrap the retriever in a `@tool`. It should retrieve the top chunks and return them as a
readable, **cited** string the agent can use.

**TODO:** implement `search_company_docs` — call `retriever.invoke(query)`, then join the
hits into a string that includes each chunk's `metadata['source']`.

In [ ]:
from langchain_core.tools import tool

@tool
def search_company_docs(query: str) -> str:
    """Search internal company docs (HR, finance, runbooks) for an answer."""
    # TODO: hits = retriever.invoke(query)
    # TODO: join hits into a cited string using h.metadata['source'] and h.page_content
    ...

In [ ]:
print(search_company_docs.invoke({"query": "PTO policy"}))

---
## Step 3 · Web search tool (from v2)

The external-knowledge tool, unchanged — Seltz wrapped as a `@tool`.

In [ ]:
from seltz import Seltz

_seltz = Seltz()

@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for current, external information."""
    try:
        resp = _seltz.search(query, max_results=max_results)
    except Exception as e:
        return f"search error: {e}"
    return "\n\n".join(f"{d.url}\n{(d.content or '')[:300]}" for d in resp.documents) or "no results"

---
## Step 4 · Assemble v3

Same graph as before — now with **two knowledge sources**. The system prompt tells the agent
when to use each.

**TODO:** put both tools in the `tools` list.

In [ ]:
tools = [...]   # TODO: include search_company_docs and web_search

In [ ]:
from typing import Annotated, TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

SYSTEM_PROMPT = (
    "You are the company Knowledge Assistant. For questions about internal policy, "
    "people, or operations, use search_company_docs. For general or current external "
    "info, use web_search. Always cite what you used."
)

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state: State):
    return {"messages": [llm_with_tools.invoke([("system", SYSTEM_PROMPT)] + state["messages"])]}

def should_continue(state: State) -> str:
    return "tools" if state["messages"][-1].tool_calls else "end"

builder = StateGraph(State)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
assistant = builder.compile()
display(Image(assistant.get_graph().draw_mermaid_png()))

---
## Step 5 · Run it — watch it route

An **internal** question (→ docs tool) and an **external** one (→ web tool). Same agent, different source, chosen by the model.

In [ ]:
def ask(q):
    out = assistant.invoke({"messages": [("user", q)]})
    return out["messages"][-1].content

print(ask("How many PTO days do I get and how do I request them?"))

In [ ]:
print(ask("What is the latest major version of LangGraph?"))

### Inspect the trace — which tool did it pick?

In [ ]:
out = assistant.invoke({"messages": [("user", "Who is on-call for billing?")]})
for m in out["messages"]:
    m.pretty_print()

---
## Step 6 · Add short-term memory (→ v3.1)

Right now each `invoke` is independent — the assistant **forgets** between turns (the third wall
from Module 2). We fix that with a **checkpointer**: compile the graph with one, and pass a
**`thread_id`**. LangGraph then saves the message history per thread and reloads it each turn.

**TODO:** compile the same `builder` with `checkpointer=InMemorySaver()`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# TODO: compile `builder` again, this time passing checkpointer=InMemorySaver()
assistant_mem = ...

Now talk to it **on a thread**. Same `thread_id` → it remembers; a new `thread_id` → fresh session (this is how you keep separate users/chats apart).

In [ ]:
def chat(text, thread_id):
    cfg = {"configurable": {"thread_id": thread_id}}
    out = assistant_mem.invoke({"messages": [("user", text)]}, cfg)
    return out["messages"][-1].content

# same thread — two turns:
chat("My name is Alessandro and I own the billing service.", thread_id="user-1")
print("same thread :", chat("What service do I own?", thread_id="user-1"))

# a different thread has no memory of the above:
print("new thread  :", chat("What service do I own?", thread_id="user-2"))

> `thread_id="user-1"` recalls the earlier turn; `"user-2"` doesn't. That per-thread history is **short-term memory** — and it's exactly what the Streamlit chat UI will use to keep a conversation going (Module 8).

---
## Key takeaways
- **Retrieval is just a tool** — wrap it, and the agent *decides* when to consult internal docs.
- The assistant now has **two knowledge sources** and routes between them — no graph changes.
- v3 finally answers about *our* company — the Module 2 "no knowledge" wall is gone.
- A **checkpointer + `thread_id`** gives short-term memory — the assistant remembers within a conversation (v3.1). The "forgets" wall is gone too.

➡️ **Next:** context engineering (what actually goes in the window) and **MCP** (a standard way
to plug in whole toolsets).